In [1]:
import os
import subprocess
import logging
import sys

def source_lmod_script(script_path):
    """
    Source an Lmod/module script and import environment variables into Python safely,
    suppressing terminal warnings.
    """
    # Use a non-interactive login shell (bash -l), redirect errors
    command = f'bash -l -c "source {script_path} >/dev/null 2>&1; printenv -0"'
    
    proc = subprocess.Popen(command, stdout=subprocess.PIPE, shell=True)
    out, _ = proc.communicate()
    
    # Parse null-separated environment variables
    for env_var in out.split(b'\0'):
        if env_var:
            key, _, value = env_var.partition(b'=')
            os.environ[key.decode()] = value.decode()

# Example usage
M3_BUILD_DIR = "/home/henryi/scratch/venvs/.venv_sbi/bin/"
TUTORIAL_BUILD_DIR = M3_BUILD_DIR
source_lmod_script(f"{M3_BUILD_DIR}/setup.MaCh3.sh")
source_lmod_script(f"{TUTORIAL_BUILD_DIR}/setup.MaCh3Tutorial.sh")
os.environ["OMP_NUM_THREADS"] = "8"


my_stderr = sys.stderr = open('errors.txt', 'w')  # redirect stderr to file
get_ipython().log.handlers[0].stream = my_stderr  # log errors to new stderr
get_ipython().log.setLevel(logging.INFO)  # errors are logged at info level

cat: write error: Broken pipe
cat: write error: Broken pipe


In [2]:
from mach3sbitools.inference.sbi_interface import MaCh3SBIInterface
from mach3sbitools.mach3_interface.mach3_simulator import MaCh3Simulator
from mach3sbitools.utils.logger import MaCh3Logger, get_logger
from mach3sbitools.utils.config import TrainingConfig, PosteriorConfig

import pickle as pkl
from pathlib import Path
import uproot as ur
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import pyplot as plt
from tqdm import tqdm

import numpy as np
import matplotlib.pyplot as plt

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde
from rich.table import Table
from rich.console import Console
from tqdm import tqdm


import torch
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages

import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D
from rich.table import Table
from rich.console import Console
import itertools

logger = get_logger("mach3sbitools")
log_level='INFO'
log_file=None
MaCh3Logger("mach3sbitools", level=log_level, log_file=log_file)

In [3]:
# Plotting utils
def circular_credible_interval(sample, lo_p, hi_p, period=2 * np.pi):
    sample = np.sort(sample % period)
    n = len(sample)
    n_in = int(np.ceil((hi_p - lo_p) / 100 * n))
    sample_ext = np.concatenate([sample, sample + period])
    idx = np.argmin(sample_ext[n_in : n_in + n] - sample_ext[:n])
    lo = (sample_ext[idx] % period + np.pi) % period - np.pi
    hi = (sample_ext[idx + n_in] % period + np.pi) % period - np.pi
    return lo, hi


def get_credible_intervals(sample, is_circular=False):
    return {
        label: circular_credible_interval(sample, lo_p, hi_p)
        if is_circular
        else tuple(np.percentile(sample, [lo_p, hi_p]))
        for label, (lo_p, hi_p) in CI_LEVELS.items()
    }


def get_best_fit(sample, is_circular=False):
    """Posterior mean — minimum MSE point estimate."""
    if is_circular:
        return np.arctan2(np.mean(np.sin(sample)), np.mean(np.cos(sample)))
    return np.mean(sample)


def plot_credible_intervals(ax, bounds, color):
    for label, (lo, hi) in bounds.items():
        ax.axvline(lo, color=color, linestyle=CI_STYLES[label], alpha=0.7)
        ax.axvline(hi, color=color, linestyle=CI_STYLES[label], alpha=0.7)


def plot_best_fit(ax, best_fit, color, ypos=0.92):
    """Draw the best-fit as a vertical line with an 'x' marker near the top."""
    ymin, ymax = ax.get_ylim()
    ax.axvline(best_fit, color=color, linestyle="solid", linewidth=1.5, alpha=0.9)
    ax.plot(best_fit, ymin + ypos * (ymax - ymin),
            marker="x", markersize=8, markeredgewidth=2,
            color=color, zorder=5, clip_on=False)


def wrap_circular(arr):
    return (arr + np.pi) % (2 * np.pi) - np.pi



In [4]:
inference = MaCh3SBIInterface("Tutorial", Path("/home/henryi/sft/MaCh3Tutorial/TutorialConfigs/FitterConfig_AsimovB.yaml"),
                              cyclical_pars=['delta_cp'])

[Monitor.cpp][info] ##################################
[Monitor.cpp][info] Welcome to:  
[Monitor.cpp][info]   __  __        _____ _     ____  
[Monitor.cpp][info]  |  \/  |      / ____| |   |___ \ 
[Monitor.cpp][info]  | \  / | __ _| |    | |__   __) |
[Monitor.cpp][info]  | |\/| |/ _` | |    | '_ \ |__ < 
[Monitor.cpp][info]  | |  | | (_| | |____| | | |___) |
[Monitor.cpp][info]  |_|  |_|\__,_|\_____|_| |_|____/ 
[Monitor.cpp][info] Version: 2.3.1
[Monitor.cpp][info] ##################################
[Monitor.cpp][info] Using following CPU:
[Monitor.cpp][info] model name	: AMD EPYC 9454 48-Core Processor
[Monitor.cpp][info] cpu MHz		: 3800.110
[Monitor.cpp][info] Architecture:                         x86_64
[Monitor.cpp][info] L1d cache:                            1.5 MiB (48 instances)
[Monitor.cpp][info] L1i cache:                            1.5 MiB (48 instances)
[Monitor.cpp][info] L2 cache:                             48 MiB (48 instances)
[Monitor.cpp][info] L3 cache:         

In [5]:
print(inference.simulator.mach3_wrapper.get_data_bins())

[3505.7295253250923, 8100.470162262577, 2621.2435292737437, 2056.242554031046, 1685.888557576371, 1392.040285703836, 942.7169510398063, 782.4436188435064, 764.2047091149531, 728.8767154159154, 1300.976921034125, 1191.0355673077763, 1244.3261276365906, 1214.616372053868, 0.0, 0.0, 0.0, 9.181766835409169, 75.05486259749362, 11.878658374439333, 5.611977626806634, 2.9321656176503894, 1.4961549937648013, 0.7400396344723811, 0.5862671318793848, 0.42311760785682145, 0.2023343238004163, 0.492787901415793, 0.4759239442691899, 0.4141792738135535, 0.4120566912860349, 0.0, 0.0, 0.0]


In [6]:
input_tree = "/home/henryi/sft/MaCh3Tutorial/Test_AsimovB.root"
mach3_chain = ur.open(f"{input_tree}:posteriors").arrays(library="np")

# input_tree = "/home/henryi/sft/MaCh3Tutorial/Test_NoAdapt.root"
# mach3_chain = ur.open(f"{input_tree}:posteriors").arrays(library="np")


In [7]:
# model_path = Path(f"/home/henryi/scratch/TutorialSBI/analyses/10M_256Hid_15Tr_MAF_lr1e-4/models/256hid_20tr.ts")
model_path=Path("/home/henryi/scratch/TutorialSBI/analyses/10M_350Hid_20Tr_MAF_lr1e-5_BS1024/models/256hid_20tr.ts")
posterior_config = PosteriorConfig(
    hidden_features = 350,      # was 50 — scale up to match data volume
    num_transforms= 20,         # was 10 — fewer needed with NSF, less sequential overhead
    dropout_probability= 0.1,  # fine as-is, good regularisation for 5M samples  
    num_blocks= 2,             # fine as-is
    # NSF-specific
    # num_bins= 10,             
    # spline bins, 8-12 is the usual sweet spot
)

inference.load_posterior(model_path, posterior_config)
samples = inference.sample_posterior(500000).cpu()

2026-03-03 02:37:21 INFO     Loading autosave checkpoint from epoch 1436

2026-03-03 02:37:25 INFO     NPE created | maf | hidden=350 transforms=20 blocks=2 bins=10

2026-03-03 02:37:26 INFO     Density estimator loaded from                                                         
                             /home/henryi/scratch/TutorialSBI/analyses/10M_350Hid_20Tr_MAF_lr1e-5_BS1024/models/256
                             hid_20tr.ts

                    INFO     Sampling 500,000 points from posterior

  0%|          | 0/500000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [15]:
# Now we refine the estimate
from sbi.inference import NPE, ImportanceSamplingPosterior
from mach3sbitools.utils.device_handler import TorchDeviceHandler
from tqdm.auto import tqdm

xo = TorchDeviceHandler().to_tensor(inference.simulator.mach3_wrapper.get_data_bins())

def simulate(t):
    return inference.simulator.mach3_wrapper.get_llh(t)

def neg_loglikelihood(theta, _):
    theta_cpu = theta.cpu().numpy()
    llh = TorchDeviceHandler().to_tensor([simulate(t) for t in tqdm(theta_cpu)])
    llh[llh>0]*=-1
    return llh

inference.posterior.set_default_x(xo)

posterior_sir = ImportanceSamplingPosterior(
    potential_fn=neg_loglikelihood,
    proposal=inference.posterior,
    method="sir",
    device='cuda',
    max_sampling_batch_size=100_000
)

samples_refined = posterior_sir.sample(
    (1000000,),
    oversampling_factor=5,
    x=xo,
    max_sampling_batch_size=100_000
)


  0%|          | 0/500000 [00:00<?, ?it/s]

  0%|          | 0/500000 [00:00<?, ?it/s]

  0%|          | 0/500000 [00:00<?, ?it/s]

  0%|          | 0/500000 [00:00<?, ?it/s]

  0%|          | 0/500000 [00:00<?, ?it/s]

...

  0%|          | 0/500000 [00:00<?, ?it/s]

  0%|          | 0/500000 [00:00<?, ?it/s]

  0%|          | 0/500000 [00:00<?, ?it/s]

  0%|          | 0/500000 [00:00<?, ?it/s]

  0%|          | 0/500000 [00:00<?, ?it/s]

In [ ]:
MACH3_LABEL = "MaCh3"
MACH3_COLOR = "#0072B2"

# Add as many ML entries as you like: label -> (samples_array, color)
ML_ENTRIES = {
    "ML":         (samples,         "#E65C00"),
    "ML Refined": (samples_refined, "#23ad48"),
}

# CI_LEVELS = {"1σ": (16, 84), "2σ": (2.5, 97.5), "3σ": (0.15, 99.85), "4σ": (100-99.9937, 99.9937)}
# CI_STYLES = {"1σ": "solid", "2σ": "dashed", "3σ": "dotted", "4σ": "dashdot"}
CI_LEVELS = {"1σ": (16, 84), "3σ": (0.15, 99.85)}
CI_STYLES = {"1σ": "solid", "3σ": "dotted"}


param_names = inference.simulator.mach3_wrapper.get_parameter_names()
summary    = {}   # name -> {model: bounds}
summary_bf = {}   # name -> {model: best_fit}

with PdfPages(model_path.parent.parent / "model_comp_ref.pdf") as pdf:
    for i, name in tqdm(enumerate(param_names), total=len(param_names)):
        is_circular = name == "delta_cp"

        # Build data dict: MaCh3 first, then all ML entries
        data = {MACH3_LABEL: (mach3_chain[name][100000:], MACH3_COLOR)}
        for label, (arr, color) in ML_ENTRIES.items():
            data[label] = (arr[:, i].cpu().numpy(), color)

        if is_circular:
            data = {k: (wrap_circular(arr), color) for k, (arr, color) in data.items()}

        bounds    = {k: get_credible_intervals(arr, is_circular) for k, (arr, _) in data.items()}
        best_fits = {k: get_best_fit(arr, is_circular)           for k, (arr, _) in data.items()}

        summary[name]    = bounds
        summary_bf[name] = best_fits

        fig, ax = plt.subplots()

        # MaCh3 sets the bins; ML entries reuse them
        _, bins, _ = ax.hist(
            data[MACH3_LABEL][0], bins=100, density=True, histtype="step",
            color=MACH3_COLOR, label=f"{MACH3_LABEL} Chain, 50M step"
        )
        for label, (arr, color) in data.items():
            if label == MACH3_LABEL:
                continue
            ax.hist(arr, bins=bins, density=True, histtype="step", color=color, label=label)

        for label, (_, color) in data.items():
            plot_credible_intervals(ax, bounds[label], color)

        # Draw best-fit crosses after histograms so y-limits are established
        fig.canvas.draw()
        for label, (_, color) in data.items():
            plot_best_fit(ax, best_fits[label], color)

        # Legend: histogram entries + per-model CI + best-fit blocks
        hist_handles, _ = ax.get_legend_handles_labels()
        ci_handles = []
        for label, (_, color) in data.items():
            ci_handles.append(Line2D([], [], color="none", label=f"── {label} ──"))
            ci_handles.append(
                Line2D([], [], color=color, linestyle="solid", linewidth=1.5,
                       marker="x", markersize=6, markeredgewidth=1.5, alpha=0.9,
                       label=f"Best fit: {best_fits[label]:.4f}")
            )
            ci_handles += [
                Line2D([], [], color=color, linestyle=CI_STYLES[lbl], alpha=0.7,
                       label=f"{lbl}: [{lo:.3f}, {hi:.3f}]")
                for lbl, (lo, hi) in bounds[label].items()
            ]

        ax.legend(handles=hist_handles + ci_handles, loc="upper left",
                  bbox_to_anchor=(1, 1), fontsize=7)
        ax.set(xlabel=name, ylabel="Posterior Density",
               title=f"Comparison of posterior for {name}")

        pdf.savefig(fig, bbox_inches="tight")
        plt.show()
        plt.close(fig)

# Summary table
console = Console()
table = Table(title="Posterior Credible Intervals", show_header=True, header_style="bold")
for col, kw in [
    ("Parameter",         {"style": "cyan", "no_wrap": True}),
    ("Model",             {"style": "magenta"}),
    ("Best Fit",          {"justify": "center"}),
    ("1σ [16, 84]",       {"justify": "center"}),
    ("2σ [2.5, 97.5]",    {"justify": "center"}),
    ("3σ [0.15, 99.85]",  {"justify": "center"}),
]:
    table.add_column(col, **kw)

for name, bounds in summary.items():
    for j, (model, model_bounds) in enumerate(bounds.items()):
        table.add_row(
            name if j == 0 else "",
            model,
            f"{summary_bf[name][model]:.4f}",
            *[f"[{lo:.4f}, {hi:.4f}]" for lo, hi in model_bounds.values()],
        )
    table.add_section()

console.print(table)

In [ ]:

def autocorrelation(x):
    x = np.asarray(x)
    x = x - np.mean(x)              # remove mean
    result = np.correlate(x, x, mode='full')
    result = result[result.size // 2:]  # keep non-negative lags
    return result / result[0]       # normalize

plt.plot(autocorrelation(mach3_chain['xsec_8']))

plt.show()
